# 📊 Práctica — Inteligencia de Negocios
## Análisis y Predicción de Propinas con Machine Learning

---

**Materia:** Inteligencia de Negocios  
**Dataset:** Tips (Propinas en un restaurante)  
**Objetivo:** Analizar qué factores influyen en la propina que deja un cliente y construir un modelo simple para predecirla.

---

### 🗂️ ¿Qué contiene el dataset?

| Variable | Descripción | Tipo |
|---|---|---|
| `total_bill` | Total de la cuenta | Numérico |
| `tip` | Propina dejada 🎯 | Numérico |
| `sex` | Género del cliente | Categórico |
| `smoker` | ¿Es fumador? | Categórico |
| `day` | Día de la semana | Categórico |
| `time` | Comida o Cena | Categórico |
| `size` | Número de personas en la mesa | Numérico |

> 🎯 **Pregunta de negocio:** ¿Qué factores influyen en la propina que deja un cliente?

---
## 🔹 Paso 1 — Cargar librerías y el dataset

Primero importamos las herramientas que vamos a usar durante toda la práctica.

In [ ]:
# ── Librerías de análisis de datos ──────────────────────────────────────────
import pandas as pd          # Para manejar tablas de datos (DataFrames)
import numpy as np           # Para operaciones matemáticas

# ── Librerías de visualización ───────────────────────────────────────────────
import matplotlib.pyplot as plt   # Para crear gráficas
import seaborn as sns             # Para gráficas más estéticas y rápidas

# ── Librerías de Machine Learning ────────────────────────────────────────────
from sklearn.model_selection import train_test_split   # Para dividir los datos
from sklearn.tree import DecisionTreeRegressor, plot_tree  # Nuestro modelo
from sklearn.metrics import mean_absolute_error, r2_score  # Para evaluar

# ── Cargar el dataset ────────────────────────────────────────────────────────
# El dataset 'tips' ya viene incluido en la librería seaborn
df = sns.load_dataset('tips')

print('✅ Dataset cargado correctamente')
print(f'   Filas: {df.shape[0]} | Columnas: {df.shape[1]}')

---
## 🔹 Paso 2 — Exploración inicial del dataset

Antes de cualquier análisis, necesitamos **conocer nuestros datos**: cuántos registros hay, qué tipos de datos contiene y si hay valores faltantes.

In [ ]:
# Mostrar las primeras 5 filas del dataset
# Esto nos da una vista rápida de cómo se ven los datos
print('=== Primeras 5 filas del dataset ===')
df.head()

In [ ]:
# Información general: tipos de datos y valores no nulos por columna
# 'object' significa texto/categoría, 'float64' e 'int64' son números
print('=== Información general del dataset ===')
df.info()

In [ ]:
# Estadísticas descriptivas de las columnas numéricas
# count = total de registros
# mean  = promedio
# std   = qué tanto varían los datos
# min / max = valores extremos
print('=== Estadísticas descriptivas ===')
df.describe()

In [ ]:
# Verificar si hay valores nulos (datos faltantes)
# Si el resultado es 0 en todas las columnas, el dataset está limpio
print('=== Valores nulos por columna ===')
print(df.isnull().sum())

# También revisamos cuántos valores únicos tiene cada columna categórica
print('\n=== Valores únicos en columnas categóricas ===')
for col in ['sex', 'smoker', 'day', 'time']:
    print(f'  {col}: {df[col].unique()}')

> ### 💡 Preguntas para responder antes de continuar
> 1. ¿Cuántos registros tiene el dataset?
> 2. ¿Hay valores nulos? ¿Necesitamos limpiar los datos?
> 3. ¿Cuál es la propina **mínima**, **máxima** y **promedio**?
> 4. ¿Cuántos días de la semana están registrados?

---
## 🔹 Paso 3 — Análisis Exploratorio (EDA)

El EDA (**Exploratory Data Analysis**) es la etapa más importante en BI.  
Aquí buscamos **patrones, tendencias y relaciones** entre las variables **antes** de usar cualquier modelo.

In [ ]:
# ── Gráfica 1: Distribución de las propinas ──────────────────────────────────
# Un histograma muestra cómo se distribuyen los valores de una variable
# Si la mayoría de barras están a la izquierda, la mayoría de propinas son bajas

plt.figure(figsize=(7, 4))
plt.hist(df['tip'], bins=20, color='steelblue', edgecolor='white')
plt.title('Distribución de Propinas', fontsize=14)
plt.xlabel('Propina ($)')
plt.ylabel('Frecuencia (número de veces)')

# Línea vertical en el promedio
plt.axvline(df['tip'].mean(), color='red', linestyle='--',
            label=f"Promedio: ${df['tip'].mean():.2f}")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfica 2: Propina por día de la semana ───────────────────────────────────
# Un boxplot muestra la distribución de los datos:
#   - La línea del centro es la mediana
#   - La caja abarca el 50% central de los datos
#   - Los puntos fuera son valores atípicos (outliers)

plt.figure(figsize=(7, 4))
sns.boxplot(
    data=df,
    x='day',
    y='tip',
    palette='pastel',
    order=['Thur', 'Fri', 'Sat', 'Sun']   # Orden lógico de los días
)
plt.title('Propina por Día de la Semana', fontsize=14)
plt.xlabel('Día')
plt.ylabel('Propina ($)')
plt.tight_layout()
plt.show()

In [ ]:
# ── Gráfica 3: ¿La cuenta total influye en la propina? ────────────────────────
# Un scatter plot muestra si dos variables tienen relación
# Si los puntos forman una línea diagonal ascendente → relación positiva

plt.figure(figsize=(7, 4))
plt.scatter(df['total_bill'], df['tip'], alpha=0.5, color='coral', edgecolors='white')
plt.title('¿A mayor cuenta, mayor propina?', fontsize=14)
plt.xlabel('Total de la Cuenta ($)')
plt.ylabel('Propina ($)')
plt.tight_layout()
plt.show()

# Calcular la correlación numérica entre las dos variables
# Va de -1 a 1: valores cercanos a 1 = correlación positiva fuerte
correlacion = df['total_bill'].corr(df['tip'])
print(f'Correlación entre cuenta total y propina: {correlacion:.2f}')

In [ ]:
# ── Gráfica 4: Propina promedio por hora del día y género ────────────────────
# groupby agrupa los datos, como un 'GROUP BY' de SQL
# unstack() convierte una columna en columnas separadas para graficar

pivot = df.groupby(['time', 'sex'])['tip'].mean().unstack()

print('Propina promedio por hora y género:')
print(pivot.round(2))

pivot.plot(kind='bar', figsize=(7, 4), color=['salmon', 'steelblue'], edgecolor='white')
plt.title('Propina Promedio: Hora del Día y Género', fontsize=14)
plt.xlabel('')
plt.ylabel('Propina Promedio ($)')
plt.xticks(rotation=0)
plt.legend(title='Género')
plt.tight_layout()
plt.show()

> ### 💡 Preguntas para discutir en clase
> 1. ¿En qué día de la semana se dejan **más propinas**? ¿Tiene sentido desde el punto de vista del negocio?
> 2. La correlación entre cuenta total y propina, ¿es fuerte o débil?
> 3. ¿Quiénes dejan más propina en promedio, hombres o mujeres?
> 4. ¿Hay diferencia significativa entre comida (Lunch) y cena (Dinner)?

---
## 🔹 Paso 4 — Preparar los datos para el modelo

Los modelos de Machine Learning **solo entienden números**.  
Por eso debemos convertir las variables categóricas (texto) en números.  
La técnica se llama **One-Hot Encoding**.

In [ ]:
# ── One-Hot Encoding ──────────────────────────────────────────────────────────
# Convierte cada categoría en una columna de 0s y 1s
# Ejemplo: la columna 'sex' se convierte en 'sex_Male' (0 = Female, 1 = Male)
# drop_first=True evita redundancia (si no es Male, entonces es Female)

df_modelo = pd.get_dummies(df, columns=['sex', 'smoker', 'day', 'time'], drop_first=True)

print('Columnas después de la codificación:')
print(df_modelo.columns.tolist())

print('\nPrimeras filas del dataset codificado:')
df_modelo.head()

In [ ]:
# ── Separar variables de entrada (X) y objetivo (y) ──────────────────────────
# X = todo lo que el modelo usará para aprender (las "pistas")
# y = lo que queremos predecir (la propina)

X = df_modelo.drop(columns=['tip'])   # Todas las columnas EXCEPTO la propina
y = df_modelo['tip']                  # Solo la columna de propina

print(f'Variables de entrada (X): {X.shape[1]} columnas')
print(f'Variable objetivo    (y): propina ($)')

# ── Dividir en datos de entrenamiento y de prueba ─────────────────────────────
# Entrenamiento (80%): el modelo aprende con estos datos
# Prueba       (20%): evaluamos qué tan bien aprendió con datos que NO vio antes
# random_state=42 asegura que todos obtengan la misma división al correr el código

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% para prueba
    random_state=42     # Semilla para reproducibilidad
)

print(f'\nDatos de entrenamiento: {X_train.shape[0]} registros (80%)')
print(f'Datos de prueba:        {X_test.shape[0]} registros (20%)')

> ### 💡 Preguntas para reflexionar
> 1. ¿Por qué no podemos darle texto directamente al modelo?
> 2. ¿Por qué dividimos los datos en entrenamiento y prueba?
> 3. ¿Qué pasaría si evaluáramos el modelo con los mismos datos con los que aprendió?

---
## 🔹 Paso 5 — Entrenar el modelo: Árbol de Decisión

Usamos un **Árbol de Decisión** porque:
- ✅ Es visualmente interpretable (se puede dibujar y leer)
- ✅ No requiere fórmulas matemáticas complejas para explicarlo
- ✅ Funciona muy bien en contextos de negocio

El parámetro `max_depth=3` limita el árbol a **3 niveles** para que sea fácil de leer.

In [ ]:
# ── Crear y entrenar el modelo ────────────────────────────────────────────────
# max_depth=3  → máximo 3 niveles de profundidad (árbol pequeño y legible)
# random_state → para que el resultado sea reproducible

modelo = DecisionTreeRegressor(max_depth=3, random_state=42)

# .fit() es la etapa de APRENDIZAJE: el modelo analiza los datos de entrenamiento
# y aprende las reglas para predecir la propina
modelo.fit(X_train, y_train)

print('✅ Modelo entrenado correctamente')

In [ ]:
# ── Visualizar el árbol de decisión ──────────────────────────────────────────
# Cada nodo del árbol representa una pregunta (condición)
# Las hojas (nodos finales) contienen el valor predicho
# El color más intenso = valor de propina más alto

plt.figure(figsize=(20, 7))
plot_tree(
    modelo,
    feature_names=X.columns.tolist(),   # Nombres de las variables
    filled=True,                         # Colorear según el valor
    rounded=True,                        # Bordes redondeados
    fontsize=10
)
plt.title('🌳 Árbol de Decisión — Predicción de Propinas', fontsize=15)
plt.tight_layout()
plt.show()

> ### 💡 ¿Cómo leer el árbol?
> - Cada nodo contiene una **condición** (pregunta)
> - Si la condición es **verdadera** → ve a la izquierda
> - Si la condición es **falsa** → ve a la derecha
> - Al llegar a una **hoja** (nodo sin hijos) → ese es el valor predicho
>
> **Ejemplo de lectura en voz alta:**  
> *"Si la cuenta total es menor o igual a $\$X$ y el tamaño de la mesa es menor o igual a $Y$, la propina estimada es $\$Z$"*

---
## 🔹 Paso 6 — Evaluar el modelo

Ahora verificamos qué tan bien predice el modelo con los datos de **prueba** (los que nunca vio).

In [ ]:
# ── Hacer predicciones sobre el conjunto de prueba ───────────────────────────
# .predict() aplica las reglas aprendidas a datos nuevos
y_pred = modelo.predict(X_test)

# ── Calcular métricas de evaluación ──────────────────────────────────────────

# MAE (Error Absoluto Medio): promedio de cuánto se equivoca el modelo en dólares
# Ejemplo: MAE = 0.80 → el modelo se equivoca ~$0.80 en promedio
mae = mean_absolute_error(y_test, y_pred)

# RMSE (Raíz del Error Cuadrático Medio): penaliza más los errores grandes
rmse = np.sqrt(np.mean((y_test - y_pred) ** 2))

# R² (R cuadrado): qué tanto del comportamiento de la propina explica el modelo
# Va de 0 a 1: más cercano a 1 = mejor modelo
# Ejemplo: R² = 0.65 → el modelo explica el 65% de la variación en propinas
r2 = r2_score(y_test, y_pred)

print('=' * 40)
print('      📊 Resultados del Modelo')
print('=' * 40)
print(f'  Error Absoluto Medio  (MAE):  ${mae:.2f}')
print(f'  Error Cuadrático      (RMSE): ${rmse:.2f}')
print(f'  R² (poder explicativo):        {r2:.2%}')
print('=' * 40)

In [ ]:
# ── Gráfica: Valores Reales vs Valores Predichos ──────────────────────────────
# Si el modelo fuera perfecto, todos los puntos estarían sobre la línea roja
# Entre más cerca estén los puntos de la línea → mejor predicción

plt.figure(figsize=(7, 5))
plt.scatter(y_test, y_pred, alpha=0.6, color='steelblue', edgecolors='white', label='Predicciones')

# Línea diagonal = predicción perfecta (real == predicho)
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    'r--', linewidth=2, label='Predicción perfecta'
)

plt.title('Propina Real vs Propina Predicha', fontsize=14)
plt.xlabel('Propina Real ($)')
plt.ylabel('Propina Predicha ($)')
plt.legend()
plt.tight_layout()
plt.show()

---
## 🔹 Paso 7 — ¿Qué variables importan más?

Una de las ventajas del Árbol de Decisión es que podemos ver  
**qué tanto aportó cada variable** para hacer las predicciones.

In [ ]:
# ── Importancia de variables ──────────────────────────────────────��───────────
# feature_importances_ indica cuánto usó el modelo cada variable
# Los valores suman 1.0 (100%)
# Una variable con 0.0 no aportó nada al modelo

importancias = pd.Series(modelo.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=True)  # Ordenar de menor a mayor

# Mostrar la tabla
print('Importancia de cada variable:')
print(importancias.sort_values(ascending=False).apply(lambda x: f'{x:.2%}'))

# Graficar
plt.figure(figsize=(8, 5))
importancias.plot(kind='barh', color='steelblue', edgecolor='white')
plt.title('¿Qué variable influye más en la propina?', fontsize=14)
plt.xlabel('Importancia relativa')
plt.tight_layout()
plt.show()

> ### 💡 Pregunta clave de Inteligencia de Negocios
> 1. ¿Cuál es la variable **más importante** según el modelo?
> 2. ¿Coincide con lo que viste en el análisis exploratorio (EDA)?
> 3. ¿Hay variables que el modelo **ignoró completamente**? ¿Por qué crees que pasó eso?
> 4. Como gerente del restaurante, ¿qué decisiones tomarías basándote en estos resultados?

---
## 🔹 Paso 8 — Hacer una predicción con datos nuevos

Probemos el modelo con un cliente ficticio para ver cuánto dejaría de propina.

In [ ]:
# ── Crear un cliente ficticio ─────────────────────────────────────────────────
# Debemos ingresar los datos en el mismo formato que usó el modelo para aprender
# Las columnas deben ser exactamente las mismas que tiene X

cliente_nuevo = pd.DataFrame({
    'total_bill': [35.0],      # Cuenta total de $35
    'size':       [4],         # Mesa de 4 personas
    'sex_Male':   [1],         # Cliente masculino (1=Male, 0=Female)
    'smoker_Yes': [0],         # No fumador
    'day_Fri':    [0],         # No es viernes
    'day_Sat':    [1],         # Sí es sábado
    'day_Sun':    [0],         # No es domingo
    'time_Dinner':[1]          # Es en la cena
})

# Asegurarnos de que las columnas estén en el mismo orden que durante el entrenamiento
cliente_nuevo = cliente_nuevo.reindex(columns=X.columns, fill_value=0)

# Hacer la predicción
propina_estimada = modelo.predict(cliente_nuevo)[0]

print('=' * 40)
print('   🔮 Predicción para cliente nuevo')
print('=' * 40)
print(f'   Cuenta total:     ${cliente_nuevo["total_bill"].values[0]:.2f}')
print(f'   Tamaño de mesa:   {int(cliente_nuevo["size"].values[0])} personas')
print(f'   Propina estimada: ${propina_estimada:.2f}')
print('=' * 40)

---
## 🏋️ Actividades

Completa las siguientes actividades **modificando el código de las celdas anteriores**:

| # | Actividad | Pista |
|---|---|---|
| 1 | Calcula el **porcentaje de propina** sobre el total (`tip / total_bill * 100`) y analiza su distribución | Crea una nueva columna en `df` |
| 2 | ¿Qué día genera **más ingresos totales** por propinas? | Usa `groupby` y `sum()` |
| 3 | Cambia `max_depth` a **2** y luego a **5**. ¿Cómo cambia el R²? | Modifica el Paso 5 |
| 4 | Crea un cliente ficticio diferente y predice su propina | Modifica el Paso 8 |

In [ ]:
# ── Actividad 1: Porcentaje de propina ───────────────────────────────────────
# Escribe tu código aquí



In [ ]:
# ── Actividad 2: Ingresos totales por propinas según el día ──────────────────
# Escribe tu código aquí



In [ ]:
# ── Actividad 3: Cambiar la profundidad del árbol ────────────────────────────
# Prueba con max_depth = 2 y max_depth = 5
# Escribe tu código aquí



In [ ]:
# ── Actividad 4: Predice la propina de un cliente diferente ──────────────────
# Escribe tu código aquí



---
## 📌 Resumen de lo aprendido

| Etapa | ¿Qué hicimos? | Herramienta |
|---|---|---|
| **Carga de datos** | Importar y conocer el dataset | `pandas`, `seaborn` |
| **EDA** | Analizar patrones y relaciones | `matplotlib`, `seaborn` |
| **Preparación** | Codificar variables y dividir datos | `get_dummies`, `train_test_split` |
| **Modelado** | Entrenar y visualizar el árbol | `DecisionTreeRegressor`, `plot_tree` |
| **Evaluación** | Medir qué tan bueno es el modelo | `MAE`, `RMSE`, `R²` |
| **Interpretación** | Identificar variables clave | `feature_importances_` |
| **Predicción** | Estimar propinas de clientes nuevos | `.predict()` |